In [1]:
from datetime import datetime
import os
import numpy as np
import pandas as pd

# import functions
from pysrc.load_data import read_data, sort_data, transform_data, drop_data
from pysrc.load_spec import load_spec
from pysrc.summarize import summarize

In [2]:
import plotly.graph_objs as go
import plotly.offline as pyo
import plotly.subplots as psub

In [3]:
from dateutil import relativedelta

In [4]:
import random
random.seed(123)

In [5]:
pd.set_option('display.max_columns', 30)

In [6]:
def detect_frequency(date_series):
    date_diffs = date_series.diff().dropna()
    mean_diff = date_diffs.mean()
    
    if pd.Timedelta(days=27) <= mean_diff < pd.Timedelta(days=32):
        return 'monthly'
    elif pd.Timedelta(days=80) <= mean_diff < pd.Timedelta(days=100):
        return 'quarterly'
    elif pd.Timedelta(days=6) <= mean_diff <= pd.Timedelta(days=7):
        return 'weekly'
    else:
        return 'unknown'


def lag_to_fill_ragged_edges(df):
    # Iterate over each column (representing different series)
    for col in df.columns:
        series = df[col]
        
        # Check if the series has missing values at the end
        if series.iloc[-1:].isna().all():
            # Find the position of the last non-NaN value
            last_non_nan = series.last_valid_index()
            
            # If valid non-NaN is found, create a lagged version of the column
            if last_non_nan is not None:
                shift_amount = len(series) - series.index.get_loc(last_non_nan) - 1
                df[col] = series.shift(shift_amount)
    
    return df

def source_data_prep(src_fpath, Spec, country="US", series_name="GDPC1"):
    df_long = pd.read_csv(src_fpath)
    df_long["ReferenceDate"] = pd.to_datetime(df_long["ReferenceDate"])

    # Create empty dataframes for monthly, weekly, and quarterly data
    monthly_data = pd.DataFrame()
    weekly_data = pd.DataFrame()
    quarterly_data = pd.DataFrame()

    # Group by 'VariableCode' to apply the frequency detection for each group
    for var_code, group in df_long.groupby('VariableCode'):
        frequency = detect_frequency(group['ReferenceDate'])
        if frequency == 'monthly':
            monthly_data = pd.concat([monthly_data, group])
        elif frequency == 'weekly':
            weekly_data = pd.concat([weekly_data, group])
        elif frequency == 'quarterly':
            quarterly_data = pd.concat([quarterly_data, group])
            
    df_m = monthly_data.pivot(index="ReferenceDate", columns="VariableCode", values="VariableValue")
    df_m = lag_to_fill_ragged_edges(df_m)

    df_q = quarterly_data.pivot(index="ReferenceDate", columns="VariableCode", values="VariableValue")
    df_q = pd.merge(
        lag_to_fill_ragged_edges(df_q.drop(columns=[series_name])),
        df_q[[series_name]],
        left_index=True,
        right_index=True
    )
    
    core_series = pd.merge(
        df_m[list(set(df_m.columns).intersection(set(Spec["seriesid"])))],
        df_q[list(set(df_q.columns).intersection(set(Spec["seriesid"])))],
        right_index=True,
        left_index=True,
        how="left"
    ).reset_index()
    other_series = df_long.loc[~df_long["VariableCode"].isin(Spec["seriesid"])]

    vintage = os.path.basename(src_fpath).split(".")[0]

    names = ["data", "other"]
    dataframes = [core_series, other_series]
    writer = pd.ExcelWriter(os.path.join('data', country, f'{vintage}.xlsx'), engine="xlsxwriter")
    for i, frame in enumerate(dataframes):
        frame.to_excel(writer, sheet_name = names[i], index=False)
    writer.close()  # Ensure the writer is properly saved
    return vintage

In [7]:
def conv_matrix_to_df(matrix, date_col):
    df = pd.DataFrame(matrix)

    headers = df.iloc[0].values
    df.columns = headers
    df = df.drop(index=0, axis=0)
    df["ReferenceDate"] = date_col
    return df.set_index("ReferenceDate").sort_index()

In [8]:
def load_data(datafile, Spec, sample=None, load_excel=False):
    """
    Load vintage of data from file and format as structure

    Parameters:
        datafile (str): Filename of Microsoft Excel workbook file
        Spec (dict): Model specification containing SeriesID and other info
        sample (float, optional): Sample period start date in numeric form
        load_excel (bool, optional): Flag to force loading from Excel

    Returns:
        X (np.ndarray): T x N numeric array, transformed dataset
        Time (np.ndarray): T x 1 numeric array, date number with observation dates
        Z (np.ndarray): T x N numeric array, raw (untransformed) dataset
    """
    print('Loading data...')

    ext = os.path.splitext(datafile)[1]  # file extension
    idx = datafile.rfind(os.path.sep)
    datafile_mat = os.path.join(datafile[:idx], 'mat', os.path.splitext(datafile[idx + 1:])[0] + '.npz')

    if os.path.exists(datafile_mat) and not load_excel:
        # Load raw data from a NumPy formatted binary (.npz) file
        with np.load(datafile_mat, allow_pickle=True) as data:
            Z = data['Z']
            Time = data['Time']
            Mnem = data['Mnem']
    elif ext in ['.xlsx', '.xls']:
        # Read raw data from Excel file
        Z, Time, Mnem = read_data(datafile)
        # np.savez(datafile_mat, Z=Z, Time=Time, Mnem=Mnem)
    else:
        raise ValueError('Only Microsoft Excel workbook files supported.')

    # Sort data based on model specification
    Z = sort_data(Z, Mnem, Spec)
    
    # Transform data based on model specification
    X, Time, Z, header = transform_data(Z, Time, Spec)

    # Drop data not in estimation sample
    if sample is not None:
        X, Time, Z = drop_data(X, Time, Z, sample)

    # Z = np.vstack([header, Z])
    # X = np.vstack([header, X])

    return X, Time, Z, header

In [9]:
from scipy.signal import lfilter
from scipy.interpolate import splrep, splev
import numpy as np

def filter(x, k):
    """Apply a moving average filter with a window size of 2*k+1."""
    numerator = np.ones(2*k+1) / (2*k+1)
    return lfilter(numerator, [1], x)

def remNaNs_spline(X,options):
    """
    Treats NaNs in the dataset for use in Dynamic Factor Models (DFM).

    This function processes NaNs in a data matrix `X` according to five cases, 
    which are useful for running functions in the `DFM.m` file that do not 
    accept missing value inputs.

    Replication files for: 
    "Nowcasting", 2010, by Marta Banbura, Domenico Giannone, and Lucrezia Reichlin, 
    in Michael P. Clements and David F. Hendry, editors, Oxford Handbook on Economic Forecasting.

    The software can be freely used in applications. Users are kindly requested to 
    add acknowledgments to published work and cite the above reference in any resulting publications.

    Args:
        X (ndarray): Input data matrix of shape (T, n) where `T` is time and `n` is the number of series.
        options (dict): A dictionary with the following keys:
            - method (int): Determines the method for handling NaNs.
                - 1: Replaces all missing values using a filter.
                - 2: Replaces missing values after removing trailing and leading zeros 
                     (a row is 'missing' if more than 80% is NaN).
                - 3: Only removes rows with leading and closing zeros.
                - 4: Replaces missing values after removing trailing and leading zeros 
                     (a row is 'missing' if all are NaN).
                - 5: Replaces missing values using a spline and then applies a filter.
            - k (int): Used in MATLAB's filter function for the 1-D filter. 
              Controls the rational transfer function's numerator, where the 
              denominator is set to 1. The numerator takes the form 
              `ones(2*k+1, 1) / (2*k+1)`. See MATLAB's documentation for `filter()` for details.

    Returns:
        tuple:
            - X (ndarray): The processed data matrix.
            - indNaN (ndarray): A matrix indicating the location of missing values (1 for NaN).
    """
    T, N = X.shape  # Gives dimensions for data input
    k = options["k"]  # Inputted options
    method = options["method"]  # Inputted options
    indNaN = np.isnan(X)  # Returns location of NaNs
    nanLE = None
    if method == 1:   # replace all the missing values
        for i in range(N):  # loop through columns
            x = X[:, i]
            isnanx = indNaN[:, i]
            x[isnanx]  = np.nanmedian(x)  # Replace missing values series median
            x_MA = filter(np.concatenate(([x[0]] * k, x, [x[-1]] * k)), k)  # Apply filter
            x_MA = x_MA[2*k:]  # Match dimensions
            x[isnanx] = x_MA[isnanx]  # Replace missing observations with filtered values
            X[:, i] = x  # Replace vector
    elif method == 2:   # replace missing values after removing leading and closing zeros
        rem1 = np.sum(indNaN, axis=1) > N * 0.8  # Returns row sum for NaN values. Marks true for rows with more than 80% NaN
        nanLead = np.cumsum(rem1) == np.arange(1, T+1)
        nanEnd = np.cumsum(rem1[::-1]) == np.arange(1, T+1)
        nanEnd = nanEnd[::-1]  # Reverses nanEnd
        nanLE = nanLead | nanEnd

        X = X[~nanLE, :]  # Remove leading and trailing NaN rows
        indNaN = np.isnan(X)  # Index for missing values

        # Loop for each series
        for i in range(N):
            x = X[:, i]
            isnanx = np.isnan(x)
            t1 = np.where(~isnanx)[0][0]  # First non-NaN entry
            t2 = np.where(~isnanx)[0][-1]  # Last non-NaN entry

            # Interpolates without NaN entries in beginning and end
            tck = splrep(np.where(~isnanx)[0], x[~isnanx], s=0)
            x[t1:t2+1] = splev(np.arange(t1, t2+1), tck)
            isnanx = np.isnan(x)

            x[isnanx] = np.nanmedian(x)  # Replace NaNs with the median

            # Apply filter
            x_MA = filter(np.concatenate(([x[0]] * k, x, [x[-1]] * k)), k)
            x_MA = x_MA[2*k:]
            x[isnanx] = x_MA[isnanx]
            X[:, i] = x

    elif method == 3:  # Only remove rows with leading and closing zeros
        rem1 = np.sum(indNaN, axis=1) == N
        nanLead = np.cumsum(rem1) == np.arange(1, T+1)
        nanEnd = np.cumsum(rem1[::-1]) == np.arange(1, T+1)
        nanEnd = nanEnd[::-1]
        nanLE = nanLead | nanEnd

        # Remove leading and trailing NaN rows
        X = X[~nanLE, :]
        indNaN = np.isnan(X)

    elif method == 4:  # Remove rows with leading and closing zeros & replace missing values
        rem1 = np.sum(indNaN, axis=1) == N
        nanLead = np.cumsum(rem1) == np.arange(1, T+1)
        nanEnd = np.cumsum(rem1[::-1]) == np.arange(1, T+1)
        nanEnd = nanEnd[::-1]
        nanLE = nanLead | nanEnd

        # Remove leading and trailing NaN rows
        X = X[~nanLE, :]
        indNaN = np.isnan(X)

        for i in range(N):
            x = X[:, i]
            isnanx = np.isnan(x)
            t1 = np.where(~isnanx)[0][0]
            t2 = np.where(~isnanx)[0][-1]

            # Interpolation
            tck = splrep(np.where(~isnanx)[0], x[~isnanx], s=0)
            x[t1:t2+1] = splev(np.arange(t1, t2+1), tck)
            isnanx = np.isnan(x)

            x[isnanx] = np.nanmedian(x)  # Replace NaNs with the median
            
            # Apply filter
            x_MA = filter(np.concatenate(([x[0]] * k, x, [x[-1]] * k)), k)
            x_MA = x_MA[2*k:]
            x[isnanx] = x_MA[isnanx]
            X[:, i] = x

    elif method == 5:  # Replace missing values
        indNaN = np.isnan(X)
        for i in range(N):
            x = X[:, i]
            isnanx = np.isnan(x)
            t1 = np.where(~isnanx)[0][0]
            t2 = np.where(~isnanx)[0][-1]

            # Interpolation
            tck = splrep(np.where(~isnanx)[0], x[~isnanx], s=0)
            x[t1:t2+1] = splev(np.arange(t1, t2+1), tck)
            isnanx = np.isnan(x)

            x[isnanx] = np.nanmedian(x)  # Replace NaNs with the median

            # Apply filter
            x_MA = filter(np.concatenate(([x[0]] * k, x, [x[-1]] * k)), k)
            x_MA = x_MA[2*k:]
            x[isnanx] = x_MA[isnanx]
            X[:, i] = x

    return X, indNaN, nanLE


In [10]:
src_fpath = "./data/0_art_vintages/2017-01-01.csv"

In [11]:
## User inputs.
src_fpath = "./data/0_art_vintages/2017-01-01.csv"
country = 'US';         # United States macroeconomic data
sample_start = datetime.strptime('2000-01-01', '%Y-%m-%d'); # estimation sample

## Load model specification and dataset.
# Load model specification structure `Spec`
Spec = load_spec('Spec_US_example.xls');
# Parse `Spec`
SeriesID, SeriesName, Units, UnitsTransformed, Frequency = Spec['seriesid'], Spec['seriesname'], Spec['units'], Spec['unitstransformed'], Spec['frequency']

vintage = source_data_prep(src_fpath=src_fpath, Spec=Spec, country=country)
# vintage = '2016-06-29'; # vintage dataset to use for estimation

# Load data
datafile = os.path.join('data', country, f'{vintage}.xls') if os.path.exists(os.path.join('data', country, f'{vintage}.xls')) else os.path.join('data', country, f'{vintage}.xlsx');
X, Time, Z, header = load_data(datafile, Spec, sample_start);
summarize(X.astype(float),Time,Spec,vintage); # summarize data

/Users/ejowik001/Desktop/Github/Nowcasting/pysrc/load_spec.py:41: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Table 1: Model specification
              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction Spen

In [12]:
source_data = pd.DataFrame(Z, columns=header, index=Time)
date_ranges = [source_data.apply(lambda col: col.first_valid_index()).max(), None]

In [13]:
# Prepare data -----------------------------------------------------------
# df = pd.DataFrame(X, columns=header, index=Time)
# x_est, y_est = df.drop(columns=["GDPC1"]).values[list(Time).index(date_ranges[0]):], df[["GDPC1"]].values[list(Time).index(date_ranges[0]):]

x_est = X[list(Time).index(date_ranges[0]):]
time_est = Time[list(Time).index(date_ranges[0]):]

# Mx = np.nanmean(x_est, axis=0)
# Wx = np.nanstd(x_est, axis=0)
# xNaN = (x_est - Mx) / Wx  # Standardize series
xNaN = x_est

optNaN = {"method": 4, "k": 3}
x_est, _, nanLE = remNaNs_spline(xNaN, optNaN)

In [14]:
monthly_gdp = pd.DataFrame(x_est, columns=header, index=time_est)[["GDPC1"]]
quarterly_gdp = pd.DataFrame(X[list(Time).index(date_ranges[0]):], columns=header, index=time_est)[["GDPC1"]]

In [15]:
X_df = pd.DataFrame(data=x_est, columns=header, index=time_est)
Z_df = pd.DataFrame(data=Z, columns=header, index=Time)

fig = psub.make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                            subplot_titles=("Raw Observed Data", "Transformed data"))

## Plot raw and transformed data.
# Industrial Production (INDPRO) <fred.stlouisfed.org/series/INDPRO>
series_name = "GDPC1"
idxSeries = SeriesID.index(series_name)
# Plot raw observed data
trace1 = go.Scatter(
    x=Z_df.sort_index()[series_name].dropna().pct_change(4).index,
    y=Z_df.sort_index()[series_name].dropna().pct_change(4),
    mode="lines+markers" if Z_df[series_name].isna().sum() else "lines",
    name='Raw Observed Data',
    line=dict(color="#000000", width=1),  # #7BCC62 / #68b562 / #7BB562
    marker={"size": 4, "symbol": "diamond"},
)
fig.add_trace(trace1, row=1, col=1)

# Plot transformed data
trace2 = go.Scatter(
    x=X_df.index,
    y=X_df[series_name],
    mode='lines',
    name='Transformed Data',
    line=dict(color="#BDC1D6", width=1),  # #7BCC62 / #68b562 / #7BB562
)
fig.add_trace(trace2, row=2, col=1)
fig.update_layout(
    height=600,
    width=800,
    showlegend=False,
    title_text=series_name,
    plot_bgcolor="white",
)
fig.update_xaxes(range=[Time[0], Time[-1]], row=1, col=1, gridcolor="lightgrey")
fig.update_yaxes(
    # title_text=Units[idxSeries],
    title_text="Percentage change Year-over-Year",
    row=1,
    col=1,
    gridcolor="lightgrey"
    )

fig.update_xaxes(range=[Time[0], Time[-1]], title_text='Time', row=2, col=1, gridcolor="lightgrey")
fig.update_yaxes(title_text=UnitsTransformed[idxSeries], row=2, col=1, gridcolor="lightgrey")
fig.show()

/var/folders/3k/vh6dl_9j30z3n567nqndm7tw0000gp/T/ipykernel_84296/816721237.py:13: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/var/folders/3k/vh6dl_9j30z3n567nqndm7tw0000gp/T/ipykernel_84296/816721237.py:14: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [16]:
# Prepare data -----------------------------------------------------------
idx_iM = [i for i in range(len(Spec["seriesid"])) if Spec["frequency"][i] == "m"]
df_m = pd.DataFrame(data=x_est[:, idx_iM], columns=header[idx_iM], index=time_est)

time_index = pd.DatetimeIndex(Time)
quarterly_index = time_index.to_period('Q')
monthly_index = pd.period_range(start=date_ranges[0], end=quarterly_index.max(), freq='M').to_timestamp()
df_m = df_m.reindex(monthly_index).dropna(how="all")
df_xNaN = pd.DataFrame(xNaN[:, idx_iM], columns=header[idx_iM], index=time_est)
df_m = df_m.fillna(df_xNaN)

idx_iQ = [i for i in range(len(Spec["seriesid"])) if Spec["frequency"][i] == "q"]
df_q = pd.DataFrame(data=x_est[:, idx_iQ], columns=header[idx_iQ], index=time_est)  # upsampled data (QS -> M)
# xNaN_sQ = pd.DataFrame(
#     X[:, idx_iQ], columns=header[idx_iQ], index=Time
# ).loc[pd.date_range(start=Time[0], end=Time[-1].date(), freq='QS')]
# x_est_q, _, nanLE_q = remNaNs_spline(xNaN_sQ.to_numpy(), options=optNaN)
# df_q = pd.DataFrame(x_est_q, columns=header[idx_iQ], index=xNaN_sQ.index[~nanLE_q])


In [17]:
print("Transformed monthly series:")
display(df_m.tail(5))
print("Transformed quarterly series:")
display(df_q.tail(5))
print("Raw observed data:")
display(Z_df.tail(5))

# target reference dates: 
# "20XX-01-01", "20XX-04-01", "20XX-07-01", "20XX-10-01"
reference_date = df_q[series_name].index.max() # + relativedelta.relativedelta(months=3)
date_ranges[1] = reference_date
print(f"Reference date: {reference_date}, estimation date ranges: {date_ranges}")

Transformed monthly series:


,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI
2016-11-01,164.0,-79.0,0.208942,-4.733436,0.151938,-0.2,-12.954545,-0.191036,0.219479,-2.004101,1.210683,0.800133,-0.082508,0.181920,0.114513,0.206956,-48.0,-2.000000e-01,-0.170302,0.082850,1.9,0.291296,8.7
2016-12-01,155.0,44.0,0.256814,-0.855858,0.984490,0.1,10.966057,0.770657,0.126706,-0.216204,1.093132,1.509607,0.412882,0.220072,0.042893,0.061958,16.0,5.000000e-01,0.820650,0.413907,9.6,0.221705,19.7
2017-01-01,216.0,-92.0,0.550611,2.436804,0.528669,0.1,-2.666667,-0.291219,0.118734,2.698460,1.639465,-0.212091,0.575658,0.307984,0.142916,0.177684,65.0,-3.000000e-01,0.338848,0.247321,7.5,0.398700,23.6
2017-02-01,219.0,86.0,0.122052,2.301167,-0.261887,-0.1,4.995971,0.061081,-0.070220,0.784236,2.300306,-0.429037,0.408831,0.205756,0.280072,0.421026,-77.0,-1.387779e-16,0.332452,0.328947,20.1,-0.210088,43.3
2017-03-01,98.0,118.0,-0.287986,0.686674,-0.216375,-0.2,-6.753645,0.548721,0.180357,0.187001,-1.768286,0.758717,-0.162866,-0.121767,0.191234,0.130239,51.0,4.000000e-01,0.305783,0.245902,15.6,-0.074456,32.8


Transformed quarterly series:


,GDPC1,ULCNFB
2016-11-01,1.297413,-1.375948
2016-12-01,0.756833,-1.491787
2017-01-01,0.694108,1.664405
2017-02-01,1.681397,0.440734
2017-03-01,1.818246,0.790457


Raw observed data:


,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI,GDPC1,ULCNFB
2016-11-01,145170,5587,242.199,228192,466028,4.6,1149,102.9771,12785.5,186398,228975,1173749,121.1,249.464,111.906,111.365,1212,75.5,1807227,120.8,1.9,11637.1,8.7,NaN,NaN
2016-12-01,145325,5631,242.821,226239,470616,4.7,1275,103.7707,12801.7,185995,231478,1191468,121.6,250.013,111.954,111.434,1228,76,1822058,121.3,9.6,11662.9,19.7,NaN,NaN
2017-01-01,145541,5539,244.158,231752,473104,4.8,1241,103.4685,12816.9,191014,235273,1188941,122.3,250.783,112.114,111.632,1293,75.7,1828232,121.6,7.5,11709.4,23.6,16842.4,111.212
2017-02-01,145760,5625,244.456,237085,471865,4.7,1303,103.5317,12807.9,192512,240685,1183840,122.8,251.299,112.428,112.102,1216,75.7,1834310,122,20.1,11684.8,43.3,NaN,NaN
2017-03-01,145858,5743,243.752,238713,470844,4.5,1215,104.0998,12831,192872,236429,1192822,122.6,250.993,112.643,112.248,1267,76.1,1839919,122.3,15.6,11676.1,32.8,NaN,NaN


Reference date: 2017-03-01 00:00:00, estimation date ranges: [Timestamp('2001-07-01 00:00:00'), Timestamp('2017-03-01 00:00:00')]


In [18]:
df_mod_m = pd.merge(df_q, df_m, how="left", left_index=True, right_index=True)

X_df, y = df_mod_m.drop(columns=[series_name]), df_mod_m[series_name]
X, _, _ = remNaNs_spline(X_df.values, options={"method": 1, "k": 3})
X = pd.DataFrame(X, columns=X_df.columns, index=X_df.index)
# train-test split
split_dt = df_q[series_name].index.max() - relativedelta.relativedelta(months=2)

train_index, test_index = y.loc[y.index < split_dt].index, y.loc[(y.index >= split_dt) & (y.index <= reference_date)].index
X_train, y_train = X.loc[X.index < split_dt],  y.loc[y.index < split_dt]
X_test, y_test = X.loc[(X.index >= split_dt) & (X.index <= reference_date)],  y.loc[(y.index >= split_dt) & (y.index <= reference_date)]

print("X train:")
display(X_train)
print("y train:")
display(y_train)

print("X test:")
display(X_test)
print("y test:")
display(y_test)

X train:


,ULCNFB,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI
2001-07-01,-5.372532,-111.0,-270.0,-1.688239e-01,-3.527644,-0.312769,1.000000e-01,2.078240,-0.575585,0.007725,-2.545384,-0.459451,0.894186,-1.536885,0.215054,0.237005,0.188648,-28.0,-0.7,-0.756277,-0.402414,-13.3,0.008382,-12.6
2001-08-01,-4.554010,-156.0,219.0,-6.790421e-17,0.208665,0.680090,3.000000e-01,-6.167665,-0.184221,1.500706,-2.589522,-1.104445,-0.614563,-0.104058,0.160944,0.260785,-0.029421,17.0,-0.3,-0.694104,-0.202020,-8.0,0.199950,-18.1
2001-09-01,-3.167561,-241.0,-365.0,3.945885e-01,-3.731142,-1.893235,1.000000e-01,-0.319081,-0.350861,1.745956,1.151715,-0.846662,-0.353451,-0.104167,0.214247,0.058957,-0.003532,-50.0,-0.4,-0.123603,0.202429,-12.5,0.524567,-8.2
2001-10-01,-1.687672,-327.0,-159.0,-2.807412e-01,4.513079,6.707112,3.000000e-01,-1.408451,-0.448628,-0.730847,-7.662287,-3.081513,-1.186244,-2.294056,0.160342,-0.549946,-0.324912,1.0,-0.5,-0.546931,-0.707071,-14.7,-0.956887,-23.6
2001-11-01,-0.588830,-291.0,-370.0,-5.630631e-02,-4.500087,-2.597645,2.000000e-01,4.025974,-0.524608,-1.559641,0.666951,0.197220,1.018495,-1.494130,0.373533,0.715630,0.363765,85.0,-0.5,-1.568500,-0.508647,-21.5,2.396726,-17.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-08-01,5.813365,176.0,438.0,2.046703e-01,0.230009,-0.027738,-5.979129e-19,-4.433498,-0.069174,0.371554,2.036416,-0.729600,0.543524,-0.248344,0.260794,0.150980,0.053314,8.0,-0.1,0.007088,-0.825764,-5.6,0.306365,4.3
2016-09-01,3.555667,249.0,-482.0,2.566673e-01,0.322680,0.996675,1.797173e-18,-9.621993,-0.151339,0.071521,1.009418,1.196193,0.526195,0.082988,0.118783,0.195619,0.159855,73.0,-0.2,0.210192,0.333056,-1.6,-0.097772,11.6
2016-10-01,0.732883,124.0,175.0,2.854701e-01,4.963103,0.656955,-1.000000e-01,25.475285,0.179047,0.194774,1.000393,-1.137481,-0.178395,0.497512,0.146795,0.106574,0.210095,35.0,0.1,0.030225,0.165975,-9.5,0.494535,11.1
2016-11-01,-1.375948,164.0,-79.0,2.089419e-01,-4.733436,0.151938,-2.000000e-01,-12.954545,-0.191036,0.219479,-2.004101,1.210683,0.800133,-0.082508,0.181920,0.114513,0.206956,-48.0,-0.2,-0.170302,0.082850,1.9,0.291296,8.7


y train:


2001-07-01   -1.259126
2001-08-01   -0.974206
2001-09-01   -0.086589
2001-10-01    1.115917
2001-11-01    2.345506
                ...   
2016-08-01    3.425363
2016-09-01    2.867769
2016-10-01    2.079755
2016-11-01    1.297413
2016-12-01    0.756833
Name: GDPC1, Length: 186, dtype: float64

X test:


,ULCNFB,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI
2017-01-01,1.664405,216.0,-92.0,0.550611,2.436804,0.528669,0.1,-2.666667,-0.291219,0.118734,2.698460,1.639465,-0.212091,0.575658,0.307984,0.142916,0.177684,65.0,-3.000000e-01,0.338848,0.247321,7.5,0.398700,23.6
2017-02-01,0.440734,219.0,86.0,0.122052,2.301167,-0.261887,-0.1,4.995971,0.061081,-0.070220,0.784236,2.300306,-0.429037,0.408831,0.205756,0.280072,0.421026,-77.0,-1.387779e-16,0.332452,0.328947,20.1,-0.210088,43.3
2017-03-01,0.790457,98.0,118.0,-0.287986,0.686674,-0.216375,-0.2,-6.753645,0.548721,0.180357,0.187001,-1.768286,0.758717,-0.162866,-0.121767,0.191234,0.130239,51.0,4.000000e-01,0.305783,0.245902,15.6,-0.074456,32.8


y test:


2017-01-01    0.694108
2017-02-01    1.681397
2017-03-01    1.818246
Name: GDPC1, dtype: float64

In [19]:
out_dir = f'./data/1_art_data_prep/{(reference_date-relativedelta.relativedelta(months=2)).strftime("%Y-%m-%d")}'
if not os.path.exists(out_dir):
    os.mkdir(out_dir)

X_train.to_csv(os.path.join(out_dir, "X_train.csv"))
X_test.to_csv(os.path.join(out_dir, "X_test.csv"))
y_train.to_csv(os.path.join(out_dir, "y_train.csv"))
y_test.to_csv(os.path.join(out_dir, "y_test.csv"))
Z_df.to_csv(os.path.join(out_dir, "Z_df.csv"))

### Growth rates retransformation

1. $ growth\_rate = (present / past) ** (1 / n) - 1 $
2. $ present = past * (1 + growth\_rate) ** n $

In [20]:
growth_rate = y_test.resample("QS").last()
past = Z_df[series_name].loc[y_train.resample("QS").last().index].tail(1)
present = Z_df[series_name].loc[y_test.resample("QS").last().index].tail(1)

step = 3
n = step / 12

In [21]:
# Actual growth rates calculation
past = Z_df[series_name].loc[y_train.resample("QS").last().index].shift(1)
present = Z_df[series_name].loc[y_train.resample("QS").last().index]

actual_growth_rates = (present / past).dropna() ** (1 / n) - 1
print(actual_growth_rates)

2001-10-01    0.011159
2002-01-01    0.037347
2002-04-01    0.022238
2002-07-01    0.019626
2002-10-01    0.002534
                ...   
2015-10-01    0.008731
2016-01-01    0.008346
2016-04-01    0.014138
2016-07-01    0.035164
2016-10-01    0.020798
Name: GDPC1, Length: 61, dtype: object


In [22]:
# Retransformation
# present = past * (1 + growth_rate) ** n

# Actual present values
print(present)  # present values from source data
print(past * (1 + actual_growth_rates) ** n)  # present values – estimated based on raw growth rated

2001-07-01    12670.1
2001-10-01    12705.3
2002-01-01    12822.3
2002-04-01      12893
2002-07-01    12955.8
               ...   
2015-10-01    16490.7
2016-01-01      16525
2016-04-01    16583.1
2016-07-01      16727
2016-10-01    16813.3
Name: GDPC1, Length: 62, dtype: object
2001-07-01        NaN
2001-10-01    12705.3
2002-01-01    12822.3
2002-04-01    12893.0
2002-07-01    12955.8
               ...   
2015-10-01    16490.7
2016-01-01    16525.0
2016-04-01    16583.1
2016-07-01    16727.0
2016-10-01    16813.3
Name: GDPC1, Length: 62, dtype: object


In [23]:
# Smoothed growth rates and present values
estimated_growth_rates = y_train.resample("QS").last()/100
past * (1 + estimated_growth_rates) ** n  #  present values – estimated based on smoothed growth rated

2001-07-01             NaN
2001-10-01    12773.803266
2002-01-01    12792.856777
2002-04-01     12889.94623
2002-07-01     12913.00697
                  ...     
2015-10-01     16487.13467
2016-01-01    16531.820718
2016-04-01    16647.891883
2016-07-01     16700.73367
2016-10-01    16758.559453
Name: GDPC1, Length: 62, dtype: object

In [24]:
# Corrected estimates of present values (residuals added to growth rates)
growth_rate_residuals = (actual_growth_rates - estimated_growth_rates)
residuals_mean, residuals_std = growth_rate_residuals.mean(), growth_rate_residuals.std()
print(residuals_mean, residuals_std)
present_corrected1 = past * (1 + (estimated_growth_rates + np.random.normal(residuals_mean, residuals_std, len(estimated_growth_rates)))) ** n

print(((present_corrected1 - present).abs()/present).dropna().mean())
present_corrected1

-2.788017040060886e-06 0.0184544629988965
0.005234881805276895


2001-07-01             NaN
2001-10-01    12786.546842
2002-01-01    12869.520264
2002-04-01    12863.800578
2002-07-01    12795.813065
                  ...     
2015-10-01    16537.306224
2016-01-01    16427.869629
2016-04-01    16661.913538
2016-07-01    16613.338416
2016-10-01    16889.803194
Name: GDPC1, Length: 62, dtype: object

In [25]:
# Corrected estimates of present values (residuals added to present values)
pred_residuals = (past * (1 + estimated_growth_rates) ** n) - present
residuals_mean, residuals_std = pred_residuals.mean(), pred_residuals.std()
print(residuals_mean, residuals_std)

present_corrected2 = (past * (1 + estimated_growth_rates) ** n) + np.random.normal(residuals_mean, residuals_std, len(estimated_growth_rates))
print(((present_corrected2 - present).abs()/present).dropna().mean())
present_corrected2

-0.2576366837605389 68.20039164312432
0.004969778442031248


2001-07-01             NaN
2001-10-01    12745.709949
2002-01-01     12743.73123
2002-04-01    12857.032015
2002-07-01    12986.297675
                  ...     
2015-10-01    16542.166921
2016-01-01    16617.748851
2016-04-01    16741.383299
2016-07-01    16767.263015
2016-10-01     16817.48737
Name: GDPC1, Length: 62, dtype: object

In [26]:
print((((present_corrected1 + present_corrected2) / 2 - present).abs()/present).dropna().mean())
print((present_corrected1 + present_corrected2) / 2)

0.00450266797406853
2001-07-01             NaN
2001-10-01    12766.128396
2002-01-01    12806.625747
2002-04-01    12860.416297
2002-07-01     12891.05537
                  ...     
2015-10-01    16539.736573
2016-01-01     16522.80924
2016-04-01    16701.648419
2016-07-01    16690.300716
2016-10-01    16853.645282
Name: GDPC1, Length: 62, dtype: object
